In [1]:
# 1. IMPORT LIBRARIES
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import CountVectorizer
import math
from IPython.display import display, HTML

In [2]:
# download stopwords if not downloaded
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to C:\Users\Jestoni
[nltk_data]     Andales\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [3]:
# 2. LOAD DATA
df = pd.read_csv("Fin_lab-PRProject_dataset.csv")

In [4]:
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

In [5]:
# 3. TEXT PREPROCESSING (STOPWORDS, LOWERCASE, STEMMING)
def preprocess(text):
    if not isinstance(text, str):
        return ""
    # lowercase
    text = text.lower()
    # split into tokens
    tokens = text.split()
    # remove stopwords
    tokens = [w for w in tokens if w not in stop_words]
    # stemming
    tokens = [stemmer.stem(w) for w in tokens]
    return " ".join(tokens)
    
df["clean"] = df["review"].apply(preprocess)
df

,Unnamed: 0,recommendationid,language,review,Reaction,clean
0,0,77057085,english,Is good. Do play.,0,good. play.
1,1,77052689,english,AAAAAAAA,0,aaaaaaaa
2,2,77049252,english,Fun game,1,fun game
3,3,77049089,english,"Great game, worth every penny!",0,"great game, worth everi penny!"
4,4,35101272,english,Like,0,like
...,...,...,...,...,...,...
46737,37,12790131,english,This game is full of shit. In a good way,0,game full shit. good way
46738,38,12790127,english,i like it,0,like
46739,39,12790085,english,"The same game , but you pay more money! 11/10...",1,"game , pay money! 11/10 must buy :d"
46740,40,12790015,english,10/10 i can cry evertim,1,10/10 cri evertim


In [6]:
# 4. BUILD BAG-OF-WORDS (SPARSE)
vectorizer = CountVectorizer()
bow = vectorizer.fit_transform(df["clean"])

vocab = vectorizer.get_feature_names_out()
vocab

array(['00', '000', '0000', ..., 'ﾞｰ', '𝓕𝓾𝓷𝓴𝔂', '𝓙𝓪𝓵𝓪𝓹𝓮𝓷𝓸𝓼'],
      shape=(26989,), dtype=object)

In [7]:
# 5. SELECT TARGET TERMS (STEMMED)
target_words = ["business", "making", "support", "data", "system"]

# stem targets
stemmed_targets = [stemmer.stem(w) for w in target_words]
stemmed_targets

['busi', 'make', 'support', 'data', 'system']

In [8]:
# 6. EXTRACT RAW COUNTS (NO toarray(), NO MEMORY ERROR)
raw_counts = {}

for term in stemmed_targets:
    if term in vocab:
        idx = vectorizer.vocabulary_[term]
        raw_counts[term] = bow[:, idx].toarray().flatten()
    else:
        raw_counts[term] = np.zeros(len(df), dtype=int)

raw_df = pd.DataFrame(raw_counts)
raw_df.columns = target_words
raw_df

,business,making,support,data,system
0,0,0,0,0,0
1,0,0,0,0,0
2,0,0,0,0,0
3,0,0,0,0,0
4,0,0,0,0,0
...,...,...,...,...,...
46737,0,0,0,0,0
46738,0,0,0,0,0
46739,0,0,0,0,0
46740,0,0,0,0,0


In [9]:
# 7. COMPUTE TF (TERM FREQUENCY)
df["total_words"] = df["clean"].apply(lambda x: len(x.split()))

tf_df = raw_df.div(df["total_words"], axis=0)
tf_df.head(100)

,business,making,support,data,system
0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...
95,0.0,0.0,0.0,0.0,0.0
96,0.0,0.0,0.0,0.0,0.0
97,0.0,0.0,0.0,0.0,0.0
98,0.0,0.0,0.0,0.0,0.0


In [11]:
# 8. COMPUTE DF & IDF
N = len(df)

df_values = (raw_df > 0).sum(axis=0)

idf_values = {}
for term in raw_df.columns:
    df_t = df_values[term]
    idf_values[term] = math.log(N / df_t)

idf_values

{'business': 6.625264012033879,
 'making': 2.7823490921028355,
 'support': 4.690941478150953,
 'data': 7.53352257221077,
 'system': 5.434278403234754}

In [12]:
# 9. COMPUTE TF-IDF
tfidf_df = tf_df.copy()

for term in raw_df.columns:
    tfidf_df[term] = tf_df[term] * idf_values[term]

tfidf_df

,business,making,support,data,system
0,0.0,0.0,0.000000,0.0,0.0
1,0.0,0.0,0.000000,0.0,0.0
2,0.0,0.0,0.000000,0.0,0.0
3,0.0,0.0,0.000000,0.0,0.0
4,0.0,0.0,0.000000,0.0,0.0
...,...,...,...,...,...
46737,0.0,0.0,0.000000,0.0,0.0
46738,0.0,0.0,0.000000,0.0,0.0
46739,0.0,0.0,0.000000,0.0,0.0
46740,0.0,0.0,0.000000,0.0,0.0


In [13]:
# 10. SHOW MOST IMPORTANT WORD PER DOCUMENT
important_terms = tfidf_df.idxmax(axis=1)
important_values = tfidf_df.max(axis=1)

result = pd.DataFrame({
    "document": df["review"],
    "most_important_term": important_terms,
    "tfidf_value": important_values
})

result

C:\Users\Jestoni Andales\AppData\Local\Temp\ipykernel_3140\20243111.py:2: FutureWarning: The behavior of DataFrame.idxmax with all-NA values, or any-NA and skipna=False, is deprecated. In a future version this will raise ValueError
  important_terms = tfidf_df.idxmax(axis=1)


,document,most_important_term,tfidf_value
0,Is good. Do play.,business,0.000000
1,AAAAAAAA,business,0.000000
2,Fun game,business,0.000000
3,"Great game, worth every penny!",business,0.000000
4,Like,business,0.000000
...,...,...,...
46737,This game is full of shit. In a good way,business,0.000000
46738,i like it,business,0.000000
46739,"The same game , but you pay more money! 11/10...",business,0.000000
46740,10/10 i can cry evertim,business,0.000000
